# Corrected main prefix comparison

Scope: **IPv4 overall traffic**, **native** selected-prefix membership, and fixed Raw-derived prefixes compared under **Raw**, **Strict**, and **Broad** conditions. Canonical `src_ip` and `dst_ip` retain first-observed direction; they are not initiator/responder roles.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

from mawi_global_analysis.comparison import build_comparison_flow_inclusion
from mawi_global_analysis.io import load_run

root = Path(os.environ.get('MAWI_ANALYSIS_ROOT', '.')).resolve()
dataset_id = os.environ.get('MAWI_DATASET_ID', 'fixture')
run_name = os.environ.get('MAWI_RUN_NAME', 'baseline')
run = load_run(dataset_id, run_name, root=root)
provenance = pd.DataFrame([{'dataset': dataset_id, 'run_name': run_name, 'analysis_scope': 'IPv4 overall / native prefixes / Raw-Strict-Broad', 'config_hash': run.manifest.get('config', {}).get('hash'), 'input_sha256': run.manifest.get('input', {}).get('sha256'), 'git_commit': run.manifest.get('git_commit')}])
display(provenance)

## Canonical-table derivation

All views below are derived here from canonical `flows.csv`, M5-produced run-local `flow_labels.csv`, the fixed Raw-derived prefix ledger, and its fixed `flow_prefix_membership.csv`; no plot-specific pipeline data is read. The M6-A inclusion API is applied to the full canonical flow/label set before the IPv4 overall scope is selected.

In [ ]:
# Build decisions on the complete canonical flow set; do not subset flows before M6-A validation.
inclusion = build_comparison_flow_inclusion(run.flows, run.labels)
flow_metrics = run.flows.merge(inclusion, on='flow_id', how='inner', validate='one_to_one')
ipv4_flows = flow_metrics.loc[pd.to_numeric(flow_metrics['ip_version']) == 4]
raw_flows = ipv4_flows
selected_prefixes = run.prefixes.loc[run.prefixes['selected_for_analysis'] == True, ['prefix', 'prefix_length', 'seen_as_src_prefix', 'seen_as_dst_prefix']].copy()
native_membership = run.membership.loc[run.membership['analysis_scope'] == 'native'].copy()
selected_prefix_ids = set(selected_prefixes['prefix'])
if not set(native_membership['analysis_prefix']).issubset(selected_prefix_ids):
    raise ValueError('native membership contains a prefix outside the fixed Raw-derived selected set')
prefix_flow_columns = ['flow_id', 'packet_count', 'frame_byte_count', 'ip_byte_count', 'duration', 'raw_included', 'strict_included', 'broad_included']
prefix_flows = native_membership.merge(ipv4_flows.loc[:, prefix_flow_columns], on='flow_id', how='inner', validate='many_to_one')
selected_scope_flows = raw_flows.loc[raw_flows['flow_id'].isin(native_membership['flow_id'].unique())]
display(pd.DataFrame([{'ipv4_raw_flow_count': len(raw_flows), 'selected_native_prefix_count': len(selected_prefixes), 'native_membership_rows': len(native_membership), 'unique_flows_matching_any_native_prefix': native_membership['flow_id'].nunique()}]))
display(selected_prefixes.sort_values('prefix').head(20))

## Raw / Strict / Broad comparison tables

These are comparison-ready tables only. They keep the same Raw-derived native prefix set and membership for every condition; comparative visualizations are deferred to M6-B2.

In [ ]:
conditions = pd.DataFrame({'condition': ['Raw', 'Strict', 'Broad'], 'inclusion_column': ['raw_included', 'strict_included', 'broad_included']})
count_columns = ['flow_count', 'packet_count', 'frame_byte_count']

def condition_metrics(frame, condition, inclusion_column):
    survivors = frame.loc[frame[inclusion_column]]
    return {'condition': condition, 'flow_count': len(survivors), 'packet_count': survivors['packet_count'].sum(), 'frame_byte_count': survivors['frame_byte_count'].sum(), 'median_packet_count': survivors['packet_count'].median(), 'median_frame_byte_count': survivors['frame_byte_count'].median(), 'median_duration': survivors['duration'].median()}

def safe_ratio(numerator, denominator):
    return numerator.div(denominator.where(denominator > 0))

overall_condition_summary = pd.DataFrame([condition_metrics(ipv4_flows, row.condition, row.inclusion_column) for row in conditions.itertuples(index=False)])
raw_overall = overall_condition_summary.loc[overall_condition_summary['condition'] == 'Raw'].iloc[0]
overall_removal_rows = []
for row in overall_condition_summary.loc[overall_condition_summary['condition'].isin(['Strict', 'Broad'])].itertuples(index=False):
    overall_removal_rows.append({'condition': row.condition, 'removed_flow_count': raw_overall.flow_count - row.flow_count, 'removed_flow_ratio': (raw_overall.flow_count - row.flow_count) / raw_overall.flow_count if raw_overall.flow_count else np.nan, 'removed_packet_count': raw_overall.packet_count - row.packet_count, 'removed_packet_ratio': (raw_overall.packet_count - row.packet_count) / raw_overall.packet_count if raw_overall.packet_count else np.nan, 'removed_frame_byte_count': raw_overall.frame_byte_count - row.frame_byte_count, 'removed_frame_byte_ratio': (raw_overall.frame_byte_count - row.frame_byte_count) / raw_overall.frame_byte_count if raw_overall.frame_byte_count else np.nan})
overall_removal_summary = pd.DataFrame(overall_removal_rows)

prefix_condition_index = selected_prefixes.loc[:, ['prefix']].rename(columns={'prefix': 'analysis_prefix'}).merge(conditions.loc[:, ['condition', 'inclusion_column']], how='cross')
prefix_condition_metrics = []
for row in conditions.itertuples(index=False):
    survivors = prefix_flows.loc[prefix_flows[row.inclusion_column]]
    metrics = survivors.groupby('analysis_prefix', as_index=False).agg(flow_count=('flow_id', 'size'), packet_count=('packet_count', 'sum'), frame_byte_count=('frame_byte_count', 'sum'), median_packet_count=('packet_count', 'median'), median_frame_byte_count=('frame_byte_count', 'median'), median_duration=('duration', 'median'))
    metrics['condition'] = row.condition
    prefix_condition_metrics.append(metrics)
per_prefix_condition_summary = prefix_condition_index.drop(columns='inclusion_column').merge(pd.concat(prefix_condition_metrics, ignore_index=True), on=['analysis_prefix', 'condition'], how='left', validate='one_to_one')
per_prefix_condition_summary.loc[:, count_columns] = per_prefix_condition_summary.loc[:, count_columns].fillna(0)

raw_prefix_metrics = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == 'Raw', ['analysis_prefix', *count_columns]].rename(columns={column: f'raw_{column}' for column in count_columns})
per_prefix_removal_rows = []
for condition in ['Strict', 'Broad']:
    survivors = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, ['analysis_prefix', *count_columns]]
    comparison = raw_prefix_metrics.merge(survivors, on='analysis_prefix', how='left', validate='one_to_one')
    per_prefix_removal_rows.append(pd.DataFrame({'analysis_prefix': comparison['analysis_prefix'], 'condition': condition, 'removed_flow_count': comparison['raw_flow_count'] - comparison['flow_count'], 'removed_flow_ratio': safe_ratio(comparison['raw_flow_count'] - comparison['flow_count'], comparison['raw_flow_count']), 'removed_packet_count': comparison['raw_packet_count'] - comparison['packet_count'], 'removed_packet_ratio': safe_ratio(comparison['raw_packet_count'] - comparison['packet_count'], comparison['raw_packet_count']), 'removed_frame_byte_count': comparison['raw_frame_byte_count'] - comparison['frame_byte_count'], 'removed_frame_byte_ratio': safe_ratio(comparison['raw_frame_byte_count'] - comparison['frame_byte_count'], comparison['raw_frame_byte_count'])}))
per_prefix_removal_summary = pd.concat(per_prefix_removal_rows, ignore_index=True)

overall_counts = overall_condition_summary.set_index('condition')['flow_count']
prefix_counts_by_condition = per_prefix_condition_summary.groupby('condition')['analysis_prefix'].nunique().reindex(conditions['condition'])
if not (overall_counts['Broad'] <= overall_counts['Strict'] <= overall_counts['Raw']):
    raise ValueError('overall survivor counts violate Broad <= Strict <= Raw')
if not (prefix_counts_by_condition == len(selected_prefixes)).all():
    raise ValueError('a comparison condition does not retain the complete fixed Raw-derived prefix set')
for condition in conditions['condition']:
    condition_prefixes = set(per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, 'analysis_prefix'])
    if condition_prefixes != selected_prefix_ids:
        raise ValueError('a comparison condition changed the fixed Raw-derived prefix set')
sanity_summary = pd.DataFrame([{'raw_selected_native_prefix_count': len(selected_prefixes), 'raw_table_prefix_count': prefix_counts_by_condition['Raw'], 'strict_table_prefix_count': prefix_counts_by_condition['Strict'], 'broad_table_prefix_count': prefix_counts_by_condition['Broad'], 'strict_zero_surviving_prefix_count': int((per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == 'Strict', 'flow_count'] == 0).sum()), 'broad_zero_surviving_prefix_count': int((per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == 'Broad', 'flow_count'] == 0).sum()), 'raw_overall_flow_count': overall_counts['Raw'], 'strict_overall_flow_count': overall_counts['Strict'], 'broad_overall_flow_count': overall_counts['Broad']}])

display(overall_condition_summary)
display(overall_removal_summary)
display(per_prefix_condition_summary.sort_values(['condition', 'analysis_prefix']))
display(per_prefix_removal_summary.sort_values(['condition', 'analysis_prefix']))
display(sanity_summary)

## Raw / Strict / Broad visual comparison

These views show movement in the already-constructed comparison tables. They do not reclassify flows or change the fixed Raw-derived prefix set.

In [ ]:
condition_order = ['Raw', 'Strict', 'Broad']
condition_labels = {'Raw': 'Raw', 'Strict': 'Strict removal', 'Broad': 'Broad expansion'}
median_metrics = ['median_packet_count', 'median_frame_byte_count', 'median_duration']
metric_labels = {'median_packet_count': 'Median packets per flow', 'median_frame_byte_count': 'Median frame bytes per flow', 'median_duration': 'Median duration (seconds)'}
overall_medians = overall_condition_summary.set_index('condition').reindex(condition_order)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for axis, metric in zip(axes, median_metrics):
    axis.plot(condition_order, overall_medians[metric], marker='o', linewidth=2, color='tab:blue')
    axis.set(title=f'Overall IPv4: {metric_labels[metric]}', xlabel='Condition', ylabel=metric_labels[metric])
fig.suptitle('Overall median movement across removal conditions', y=1.02)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
jitter = np.random.default_rng(0)
for axis, metric in zip(axes, median_metrics):
    for position, condition in enumerate(condition_order):
        values = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, metric].dropna()
        axis.scatter(position + jitter.uniform(-0.12, 0.12, len(values)), values, s=16, alpha=0.45, color='tab:orange', label='Selected native prefixes' if position == 0 else None)
        if not values.empty:
            axis.scatter(position, values.median(), marker='_', s=450, linewidths=2.5, color='tab:orange', zorder=2, label='Prefix median' if position == 0 else None)
        axis.scatter(position, overall_medians.loc[condition, metric], marker='D', s=70, color='tab:blue', zorder=3, label='Overall IPv4' if position == 0 else None)
    axis.set(title=metric_labels[metric], xlabel='Condition', ylabel=metric_labels[metric], xticks=range(len(condition_order)), xticklabels=[condition_labels[condition] for condition in condition_order])
    axis.legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:orange', markersize=7, label='Selected native prefixes'), Line2D([0], [0], marker='_', color='tab:orange', markersize=13, label='Prefix median'), Line2D([0], [0], marker='D', color='w', markerfacecolor='tab:blue', markersize=7, label='Overall IPv4')])
fig.suptitle('Overall median versus selected native-prefix median distribution', y=1.02)
fig.tight_layout()

### Per-prefix median change from Raw

Each point is one fixed `analysis_prefix`. Values closer to the identity line indicate less movement in that prefix median; rows with no surviving flows remain in the comparison table but have `NaN` medians and are not plotted.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 14))
for row_index, metric in enumerate(median_metrics):
    raw_values = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == 'Raw', ['analysis_prefix', metric]].rename(columns={metric: 'raw_value'})
    for column_index, condition in enumerate(['Strict', 'Broad']):
        axis = axes[row_index, column_index]
        compared_values = per_prefix_condition_summary.loc[per_prefix_condition_summary['condition'] == condition, ['analysis_prefix', metric]].rename(columns={metric: 'condition_value'})
        paired = raw_values.merge(compared_values, on='analysis_prefix', how='inner', validate='one_to_one').dropna()
        if paired.empty:
            axis.text(0.5, 0.5, 'No paired surviving-prefix medians', ha='center', va='center', transform=axis.transAxes)
            axis.plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
            axis.set(xlim=(0, 1), ylim=(0, 1))
        else:
            upper = max(paired['raw_value'].max(), paired['condition_value'].max(), 1)
            axis.scatter(paired['raw_value'], paired['condition_value'], s=18, alpha=0.6, color='tab:purple')
            axis.plot([0, upper], [0, upper], linestyle='--', color='black', linewidth=1)
            axis.set(xlim=(0, upper), ylim=(0, upper))
        axis.set(title=f'Raw vs {condition_labels[condition]}', xlabel=f'Raw {metric_labels[metric]}', ylabel=f'{condition_labels[condition]} {metric_labels[metric]}')
fig.suptitle('Per-prefix median change from Raw (identity line)', y=0.995)
fig.tight_layout()

In [ ]:
removal_metrics = ['removed_flow_ratio', 'removed_packet_ratio', 'removed_frame_byte_ratio']
removal_labels = ['Flows', 'Packets', 'Frame bytes']
removal_plot = overall_removal_summary.set_index('condition').reindex(['Strict', 'Broad'])
positions = np.arange(len(removal_metrics))
width = 0.35
fig, axis = plt.subplots(figsize=(8, 4.5))
for offset, condition in zip([-width / 2, width / 2], ['Strict', 'Broad']):
    axis.bar(positions + offset, removal_plot.loc[condition, removal_metrics], width=width, label=condition_labels[condition])
axis.set(title='Removal volume relative to Raw IPv4 traffic', xlabel='Quantity', ylabel='Raw-relative removal ratio', xticks=positions, xticklabels=removal_labels, ylim=(0, 1))
axis.legend()
fig.tight_layout()

## Raw baseline descriptive distributions

These existing Raw-only flow-length views remain a baseline descriptive reference. Raw/Strict/Broad distribution comparisons are intentionally outside M6-B2.

In [ ]:
def ecdf(values):
    ordered = np.sort(np.asarray(values, dtype=float))
    return ordered, np.arange(1, len(ordered) + 1) / len(ordered)

protocols = {'TCP': ['6', '6.0', 'tcp'], 'UDP': ['17', '17.0', 'udp']}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for protocol, values in protocols.items():
    for scope, frame, style in [('IPv4 overall', raw_flows, '-'), ('Native-prefix associated', selected_scope_flows, '--')]:
        packets = frame.loc[frame['protocol'].astype(str).isin(values), 'packet_count']
        if packets.empty:
            continue
        x, y = ecdf(packets)
        label = f'{protocol}: {scope}'
        axes[0].hist(packets, bins='auto', histtype='step', linewidth=2, linestyle=style, label=label)
        axes[1].step(x, y, where='post', linestyle=style, label=label)
        axes[2].step(x, 1 - y, where='post', linestyle=style, label=label)
axes[0].set(title='Packet-count histogram', xlabel='Packets per flow', ylabel='Flow count', xscale='log', yscale='log')
axes[1].set(title='Packet-count ECDF', xlabel='Packets per flow', ylabel='Cumulative probability', xscale='log')
axes[2].set(title='Packet-count CCDF', xlabel='Packets per flow', ylabel='Tail probability', xscale='log', yscale='log')
for axis in axes:
    axis.legend()
fig.tight_layout()

In [ ]:
# One row for every selected native prefix, including prefixes with zero canonical membership.
prefix_metrics = prefix_flows.groupby('analysis_prefix', as_index=False).agg(flow_count=('flow_id', 'size'), packet_count=('packet_count', 'sum'), frame_byte_count=('frame_byte_count', 'sum'), ip_byte_count=('ip_byte_count', 'sum'), median_packet_count=('packet_count', 'median'), median_frame_byte_count=('frame_byte_count', 'median'), median_duration=('duration', 'median'), q90_duration=('duration', lambda values: values.quantile(0.9)), q99_duration=('duration', lambda values: values.quantile(0.99)))
prefix_summary = selected_prefixes.rename(columns={'prefix': 'analysis_prefix'}).merge(prefix_metrics, on='analysis_prefix', how='left', validate='one_to_one')
zero_volume_prefixes = prefix_summary['flow_count'].isna()
prefix_summary[['flow_count', 'packet_count', 'frame_byte_count', 'ip_byte_count']] = prefix_summary[['flow_count', 'packet_count', 'frame_byte_count', 'ip_byte_count']].fillna(0)
display(pd.DataFrame([{'selected_prefixes_with_zero_canonical_membership': int(zero_volume_prefixes.sum()), 'selected_prefix_count': len(prefix_summary)}]))
display(prefix_summary.sort_values('frame_byte_count', ascending=False).head(20))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
positive_volume = prefix_summary.loc[prefix_summary['frame_byte_count'] > 0]
for axis, y_column, title in zip(axes, ['median_packet_count', 'median_frame_byte_count', 'median_duration'], ['Traffic volume vs median packets', 'Traffic volume vs median frame bytes', 'Traffic volume vs median duration']):
    axis.scatter(positive_volume['frame_byte_count'], positive_volume[y_column], alpha=0.75)
    axis.set(xscale='log', yscale='log' if (positive_volume[y_column] > 0).all() else 'linear', xlabel='Native-prefix frame bytes', ylabel=y_column, title=title)
fig.tight_layout()

In [ ]:
# The selected-prefix series is deduplicated by flow_id for this combined-scope view.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, values in [('IPv4 overall (Raw)', raw_flows['duration']), ('Any selected native prefix (Raw)', selected_scope_flows['duration'])]:
    x, y = ecdf(values)
    axes[0].step(x, y, where='post', label=label)
    axes[1].step(x, 1 - y, where='post', label=label)
axes[0].set(title='Flow-duration ECDF', xlabel='Duration (seconds)', ylabel='Cumulative probability')
axes[1].set(title='Flow-duration CCDF', xlabel='Duration (seconds)', ylabel='Tail probability', yscale='log')
for axis in axes:
    axis.legend()
fig.tight_layout()

This corrected baseline intentionally uses all selected non-overlapping native IPv4 prefixes. Prefix membership is `src_ip ∈ prefix OR dst_ip ∈ prefix`; `src_match` and `dst_match` remain observation-direction facts, not traffic-role labels.